# 🎯 [ISR03] 드론 비행로그 패턴 이상 탐지와 운용 주체 단정 방지
### Google Colab 실습 환경 — 미션 02 (isr03_m02_solution.ipynb) (정보·감시·정찰(ISR) / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/ISR03/isr03_m02_solution.ipynb)

---

## 📌 1. 프로젝트 개요 및 배경

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 정보·감시·정찰(ISR) (Level 3) |
| **학습 소요 시간** | 10개 미션 (5일 / 총 27.5시간) |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
러시아-우크라이나 전쟁에서는 상용 드론이 정찰과 공격에 폭넓게 쓰이면서, 노획하거나 수집한 드론 비행로그를 분석해 이상 비행을 찾아내는 일이 중요해졌다. 그러나 같은 상용 기체를 아군·민간·적대 세력이 모두 쓰기 때문에, 비행 특성만 보고 운용 주체를 성급하게 단정하면 오판으로 이어질 수 있다. 본 프로젝트에서는 교육용 가상 비행로그를 코드로 직접 만들어 스키마 정의, GPS 스푸핑 보정 전처리, 규칙 기반 Baseline, 군집·비지도 학습 기반 이상 탐지, 불확실성을 반영한 운용 주체 신뢰도 엔진, 분석 대시보드, 정보분석관 심층 검토(HITL), 대량 로그 부하 시험, AAR 재보정, 최종 통합까지 단계별로 구현한다. 최종 목표는 이상 비행을 놓치지 않으면서 오탐을 줄이고, 운용 주체에 관한 판단은 항상 확률·불확실성·근거와 함께 사람의 검토를 거쳐 보고되도록 설계하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

### 🛠️ 2. 실습 실행 및 산출물 저장·데이터 연계 가이드

1. **Colab / 로컬 실행 환경 안내**
   - 아래 환경 설정 셀의 `!pip install -q ...` 구문은 **Google Colab / Jupyter 노트북 전용** 매직 커맨드입니다.
   - 일반 로컬 Python(`.py`) 환경에서 실행할 때는 해당 줄을 제거하고 터미널에서 `pip install numpy pandas matplotlib scipy scikit-learn`을 먼저 실행하세요.

2. **산출물(`deliverables`) 디렉토리 구조 및 파일 저장 방법**
   - 본 노트북의 각 미션 셀은 Colab 환경에서 단독 실행이 가능하도록 인메모리 시뮬레이션으로 구성되어 있습니다.
   - 프로젝트 제출용 산출물(`data/schema/`, `docs/`, `src/preprocess/`, `src/models/`, `src/verification/`, `src/ui/`, `src/hitl/`, `tests/`, `configs/`)을 실제 파일로 저장하려면, 환경 설정 셀에 제공된 `init_deliverable_dirs()` 및 `save_deliverable(rel_path, content)` 헬퍼 함수 또는 셀 상단의 `%%writefile <파일경로>` 매직 커맨드를 사용하세요.

3. **첨부 데이터셋(`isr03_mXX_dataset.json`) 연계 활용**
   - 각 미션 셀은 외부 파일 의존성 없이도 독립 실행되도록 자체 난수 기반 가상 데이터를 생성합니다.
   - 미션별 첨부파일(`isr03_m01_dataset.json` ~ `isr03_m10_dataset.json`, `isr03_mXX_sample.md`)은 `load_mission_dataset(1)` 함수로 불러와 스키마 대조, 경계값 추가 검증, 파라미터 확장 실험에 연계 활용할 수 있습니다.

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")

# 산출물 디렉토리 초기화 및 파일 저장/첨부 데이터셋 로드 헬퍼
DELIVERABLE_DIRS = [
    'data/schema', 'docs', 'src/preprocess', 'src/models',
    'src/verification', 'src/ui', 'src/hitl', 'tests', 'configs'
]

def init_deliverable_dirs(base_dir='.'):
    """프로젝트 필수 산출물 디렉토리를 생성한다."""
    for d in DELIVERABLE_DIRS:
        os.makedirs(os.path.join(base_dir, d), exist_ok=True)

def save_deliverable(rel_path, content, base_dir='.'):
    """산출물 파일(JSON, YAML, MD, PY 등)을 지정 경로에 저장한다."""
    full_path = os.path.join(base_dir, rel_path)
    os.makedirs(os.path.dirname(full_path), exist_ok=True)
    with open(full_path, 'w', encoding='utf-8') as f:
        f.write(content if isinstance(content, str) else json.dumps(content, ensure_ascii=False, indent=2))
    return full_path

def load_mission_dataset(mission_num, base_dir='.'):
    """첨부된 미션별 가상 데이터셋(isr03_mXX_dataset.json)을 불러온다."""
    fname = f'isr03_m{int(mission_num):02d}_dataset.json'
    for candidate in [os.path.join(base_dir, fname), os.path.join(base_dir, 'uploads', 'ISR03', fname)]:
        if os.path.exists(candidate):
            with open(candidate, encoding='utf-8') as f:
                return json.load(f)
    raise FileNotFoundError(f'{fname} 파일을 찾을 수 없습니다. Colab에 업로드했는지 확인하세요.')


---

## 🎯 미션 02. 비행로그 내 GPS 스푸핑 왜곡값 보정 및 시계열 텔레메트리 전처리 모듈 구현
**소요시간**: 약 2.0시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

비행로그 내의 GPS 스푸핑 및 전자기 간섭에 따른 이상 좌표를 보정하고 시계열 비행 궤적 데이터를 정규화하는 전처리 모듈을 구현한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **GPS 스푸핑·전자기 간섭·센서 단절로 왜곡된 비행로그를 보정하고 시계열을 1초 간격으로 표준화·정규화하는 전처리 모듈을 구현한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **중복 기록 제거와 1초 재표본화, 결측 보간, 이동 중간값 기반 고도 스파이크 제거 구현하고 동작을 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 5개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `src/preprocess/isr03_preprocessor.py 및 전처리 전후 데이터셋` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 전술 환경 특유의 노이즈(스푸핑, 반사파, 센서 단절 등) 제거 타당성

#### 미션 PASS 조건

- 정량 기준: 데이터 정제 처리 속도 1,000건/초 이상, 전처리 오류율 0%


- 진행 기준: 정제 파이프라인 자동화 및 단위 테스트 케이스 5종 통과


- 검증 방법: pytest 기반 전처리 정합성 테스트 및 결측치 보정율 확인


- AI 검증 기준: AI가 제안한 보정 로직·임계값 해석을 실제 위치 오차 및 pytest 결과로 교차 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""비행로그의 GPS 스푸핑 위치 점프·표류, 전자기 간섭 고도 스파이크, 센서 단절 결측, 중복 기록을 보정하고 시계열을 1초 간격으로 표준화·정규화하는 전처리 모듈을 구현해 처리 속도와 보정 효과를 검증한다."""
import math
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# [교육용가상자료] 합성 드론 비행로그 생성기 (실제 좌표·기체·부대 정보 없음)
# 가상 구역(원점 기준 동·북 방향 m 좌표)에서 1초 간격 비행로그와 정답(이상 구간·운용 주체)을 만든다.
# ---------------------------------------------------------------------------
NFZ = {"x": 2500.0, "y": 2500.0, "r": 700.0}        # 가상 비인가 비행구역
WATCH = {"x": -2000.0, "y": 1500.0, "r": 600.0}     # 가상 관심 구역
SCENARIOS = ["sharp_turn", "hover", "dive", "deviation", "nfz_entry",
             "low_level", "rc_silent", "infiltration", "loiter", "speed_burst"]
OPERATORS = ["friendly", "civil", "hostile"]
# 시나리오별 운용 주체 분포: 일부러 크게 겹치게 설계했다(비행 특성만으로 운용 주체를 단정할 수 없다).
OPERATOR_P = {"normal": (.45, .47, .08), "survey": (.15, .82, .03), "sharp_turn": (.40, .35, .25),
              "hover": (.25, .45, .30), "dive": (.25, .25, .50), "deviation": (.30, .40, .30),
              "nfz_entry": (.10, .50, .40), "low_level": (.35, .15, .50), "rc_silent": (.30, .15, .55),
              "infiltration": (.12, .08, .80), "loiter": (.30, .20, .50), "speed_burst": (.30, .20, .50)}
PLATFORM_P = {"friendly": (.40, .55, .05), "civil": (.88, .02, .10), "hostile": (.55, .30, .15)}   # commercial/military/unknown
FIRMWARE_P = {"friendly": (.30, .65, .05), "civil": (.85, .10, .05), "hostile": (.20, .50, .30)}   # stock/custom/unknown


def _wrap(a):
    return (a + 180.0) % 360.0 - 180.0


def _bearing(p, q):
    return math.degrees(math.atan2(q[0] - p[0], q[1] - p[1])) % 360.0


def _away_from_nfz(rng, min_d):
    while True:
        c = rng.uniform(-3800, 3800, 2)
        if np.hypot(c[0] - NFZ["x"], c[1] - NFZ["y"]) > min_d:
            return c


def make_flight(fid, kind, rng, n=300, weather=None):
    """비행 1회 = (로그 DataFrame, 메타 dict). kind: normal / survey / SCENARIOS 중 하나"""
    weather = weather or str(rng.choice(["clear", "night", "windy", "rain"], p=[.4, .25, .2, .15]))
    wind = {"clear": 1.0, "night": 1.0, "windy": 2.5, "rain": 1.8}[weather]
    v0, alt0 = rng.uniform(11, 17), rng.uniform(70, 150)
    nfz_c = np.array([NFZ["x"], NFZ["y"]])
    if kind == "infiltration":                      # 원거리에서 비인가 구역 방향으로 저고도 직진
        ang = rng.uniform(0, 2 * np.pi)
        start = nfz_c + 4000 * np.array([np.cos(ang), np.sin(ang)])
        plan = np.array([start, nfz_c])
        v0, alt0 = rng.uniform(9, 11), rng.uniform(25, 35)
    else:
        if kind == "nfz_entry":                     # 비인가 구역 1.6km 밖에서 순찰하다가 진입
            ang = rng.uniform(0, 2 * np.pi)
            c = nfz_c + 1600 * np.array([np.cos(ang), np.sin(ang)])
        else:
            c = _away_from_nfz(rng, 2600)
        if kind == "survey":                        # 민간 측량형 왕복(지그재그) 경로
            w, h = rng.uniform(500, 700), 120.0
            plan = np.array([c + [(-w if i % 4 in (0, 3) else w), i // 2 * h - 360] for i in range(8)])
        else:                                       # 사각 순찰 경로(웨이포인트 4개)
            a, b, th = rng.uniform(300, 600), rng.uniform(200, 400), rng.uniform(0, np.pi)
            R = np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])
            plan = c + np.array([[-a, -b], [a, -b], [a, b], [-a, b]]) @ R.T
    s0 = int(rng.integers(80, 160))
    span = {"sharp_turn": 30, "hover": 60, "dive": 8, "deviation": 70, "nfz_entry": 120, "low_level": 120,
            "rc_silent": 120, "infiltration": n - 20, "loiter": 90, "speed_burst": 25}.get(kind, 0)
    if kind == "infiltration":
        s0 = 20
    win = (s0, s0 + span)
    turns = set(rng.choice(np.arange(s0, s0 + 30), 3, replace=False).tolist()) if kind == "sharp_turn" else set()
    # 난수를 미리 한 번에 뽑아 두고(속도 향상) 1초 단위로 기체 운동을 적분한다.
    e_h, e_v, e_a = rng.normal(0, 0.8 * wind, n), rng.normal(0, 0.3 * wind, n), rng.normal(0, 0.4 * wind, n)
    rc_arr = np.clip(rng.normal(rng.uniform(60, 92), 6, n), 0, 100)
    gust = (rng.random(n) < (0.004 if weather == "windy" else 0.0)) * rng.uniform(20, 35, n)
    burst, turn_amt = rng.uniform(23, 28), rng.choice([-1, 1], n) * rng.uniform(100, 160, n)
    side = float(rng.choice([-90, 90]))
    low_alt, rng_rc = rng.uniform(35, 55), rng.uniform(0, 20, n)          # 저고도 목표, 조종신호 두절 시 잔여 수신값
    x, y, wi = float(plan[0][0]), float(plan[0][1]), 1
    hdg, alt = _bearing((x, y), plan[1]), alt0
    max_rate = 40.0 if kind == "survey" else 12.0
    dev_target, X, Y, A, V, HD = None, np.zeros(n), np.zeros(n), np.zeros(n), np.zeros(n), np.zeros(n)
    for t in range(n):
        inw = win[0] <= t < win[1]
        tx, ty = plan[1] if kind == "infiltration" else plan[wi % len(plan)]
        spd_cmd, alt_cmd = v0, alt0
        if inw:
            if kind == "deviation":
                if dev_target is None:
                    r = math.radians(hdg + side)
                    dev_target = (x + 900 * math.sin(r), y + 900 * math.cos(r))
                tx, ty = dev_target
            elif kind == "nfz_entry":
                tx, ty = NFZ["x"], NFZ["y"]
            elif kind == "hover":
                spd_cmd = 0.0
            elif kind == "speed_burst":
                spd_cmd = burst
            elif kind == "low_level":
                alt_cmd = low_alt
            if kind in ("rc_silent", "infiltration"):
                rc_arr[t] = rng_rc[t]
        if inw and kind == "loiter":
            hdg = (hdg + 9.0 + e_h[t] / wind) % 360
        else:
            d = _wrap(math.degrees(math.atan2(tx - x, ty - y)) - hdg)
            hdg = (hdg + max(-max_rate, min(max_rate, d)) + e_h[t]) % 360
        if t in turns:
            hdg = (hdg + turn_amt[t]) % 360
        v = abs(spd_cmd + e_v[t]) if spd_cmd > 0 else abs(e_v[t] / wind)
        x += v * math.sin(math.radians(hdg)); y += v * math.cos(math.radians(hdg))
        if kind != "infiltration" and math.hypot(x - tx, y - ty) < 40 and not (inw and kind in ("deviation", "nfz_entry")):
            wi += 1
        if inw and kind == "dive":
            alt -= 9.0 + e_v[t]
        else:
            alt += max(-3.0, min(3.0, alt_cmd - alt)) + e_a[t]
        alt -= gust[t]                                     # 돌풍 하강기류(정상 비행에서도 발생)
        if kind == "dive" and t >= win[1]:
            alt0 = min(alt0, alt)
        alt = max(alt, 3.0)
        X[t], Y[t], A[t], V[t], HD[t] = x, y, alt, v, hdg
    gps_sd = {"clear": 2.0, "night": 2.0, "windy": 3.0, "rain": 4.5}[weather]     # 악천후일수록 GPS 잡음(다중경로) 증가
    df = pd.DataFrame({"t": np.arange(n), "x": X + rng.normal(0, gps_sd, n), "y": Y + rng.normal(0, gps_sd, n),
                       "alt": A + rng.normal(0, 0.5, n), "speed": V + rng.normal(0, 0.2, n),
                       "heading": (HD + rng.normal(0, 1.5, n)) % 360, "rc": rc_arr.round(1),
                       "sats": rng.integers(10, 15, n), "hdop": rng.uniform(0.6, 1.2, n).round(2),
                       "x_true": X, "y_true": Y})
    df["anomaly"] = (df["t"] >= win[0]) & (df["t"] < win[1]) & (kind in SCENARIOS)
    op = OPERATORS[int(rng.choice(3, p=OPERATOR_P[kind]))]
    meta = {"id": fid, "kind": kind, "weather": weather, "operator": op, "plan": plan, "window": win if kind in SCENARIOS else None,
            "platform": ["commercial", "military", "unknown"][int(rng.choice(3, p=PLATFORM_P[op]))],
            "firmware": ["stock", "custom", "unknown"][int(rng.choice(3, p=FIRMWARE_P[op]))]}
    return df, meta


def corrupt(df, rng, spoof=True, emi=True, drop=True, dup=True):
    """전장 잡음 주입: GPS 스푸핑(위치 점프·표류), 전자기 간섭(고도 스파이크), 센서 단절(결측), 중복 기록."""
    d = df.copy()
    d["spoof_true"] = False
    if spoof:
        a = int(rng.integers(40, len(d) - 60)); L = int(rng.integers(25, 45))
        th = rng.uniform(0, 2 * np.pi)
        drift = rng.uniform(22, 28) * np.arange(L)[:, None] * np.array([np.cos(th), np.sin(th)])
        jump = rng.uniform(250, 400) * np.array([np.cos(th + 1), np.sin(th + 1)])
        d.loc[a:a + L - 1, ["x", "y"]] += drift + jump
        d.loc[a:a + L - 1, "spoof_true"] = True
    if emi:
        idx = rng.choice(len(d), 6, replace=False)
        d.loc[idx, "alt"] += rng.choice([-1, 1], 6) * rng.uniform(60, 150, 6)
    if drop:
        a = int(rng.integers(10, len(d) - 20))
        d.loc[a:a + int(rng.integers(4, 9)), ["x", "y", "alt", "speed", "heading"]] = np.nan
        d = d.drop(index=rng.choice(len(d), 5, replace=False))
    if dup:
        d = pd.concat([d, d.sample(5, random_state=int(rng.integers(1e6)))]).sort_values("t", kind="stable")
    return d.reset_index(drop=True)


def make_dataset(n=200, seed=7, anomaly_ratio=0.5, scenarios=SCENARIOS):
    rng = np.random.default_rng(seed)
    out = []
    for i in range(n):
        if i % 10 < anomaly_ratio * 10:
            kind = scenarios[(i // 10 * int(anomaly_ratio * 10) + i % 10) % len(scenarios)]
        else:
            kind = "survey" if i % 10 == 9 else "normal"
        df, meta = make_flight(f"FL{i:04d}", kind, rng)
        out.append((df, meta))
    return out
# ---------------------------------------------------------------------------

# ---- 전처리·규칙 탐지 공통 함수 ----
SPOOF_THR = {"clear": 12.0, "night": 12.0, "windy": 15.0, "rain": 19.0}   # 기상별 GPS-추측항법 불일치 허용치(m)


def clean(df, weather="clear"):
    """전처리: 중복 제거·정렬 -> 1초 재표본화 -> 결측 보간 -> 고도 스파이크 제거 -> GPS 스푸핑 구간 추측항법 보정."""
    d = df.drop_duplicates("t").sort_values("t").set_index("t")
    d = d.reindex(np.arange(int(d.index.min()), int(d.index.max()) + 1))
    n_missing = int(d["x"].isna().sum())
    num = ["x", "y", "alt", "speed", "rc", "sats", "hdop"]
    d[num] = d[num].interpolate(limit_direction="both")
    hd = d["heading"].dropna()                                   # 방위는 0/360 경계를 풀어서 보간
    un = pd.Series(np.degrees(np.unwrap(np.radians(hd.to_numpy(float)))), index=hd.index)
    d["heading"] = un.reindex(d.index).interpolate(limit_direction="both").to_numpy() % 360
    med = d["alt"].rolling(5, center=True, min_periods=1).median()
    spike = (d["alt"] - med).abs() > 25
    d.loc[spike, "alt"] = med[spike]
    x, y = d["x"].to_numpy(float), d["y"].to_numpy(float)
    r = np.radians(d["heading"].to_numpy(float)); v = d["speed"].to_numpy(float)
    dr = np.c_[v * np.sin(r), v * np.cos(r)]                  # 보고 속도·방위로 계산한 1초 이동량(추측항법)
    xc, yc, flag = x.copy(), y.copy(), np.zeros(len(x), bool)
    thr = SPOOF_THR.get(weather, 12.0)
    ok_run = 0
    for i in range(1, len(x)):
        px, py = xc[i - 1] + dr[i, 0], yc[i - 1] + dr[i, 1]          # 추측항법 예상 위치
        step_ok = np.hypot(x[i] - x[i - 1] - dr[i, 0], y[i] - y[i - 1] - dr[i, 1]) <= thr
        ok_run = ok_run + 1 if step_ok else 0
        if flag[i - 1]:
            # 스푸핑 구간: GPS가 예상 위치로 돌아오거나, GPS 이동량이 5초 연속 물리적으로 맞으면 GPS로 복귀
            if np.hypot(x[i] - px, y[i] - py) < thr * 2 or ok_run >= 5:
                xc[i], yc[i] = x[i], y[i]
            else:
                xc[i], yc[i], flag[i] = px, py, True
        elif not step_ok:
            xc[i], yc[i], flag[i] = px, py, True                     # GPS 이동량이 물리적으로 맞지 않음 -> 스푸핑 의심
        else:
            xc[i], yc[i] = xc[i - 1] + x[i] - x[i - 1], yc[i - 1] + y[i] - y[i - 1]
    d["x"], d["y"], d["spoof_flag"] = xc, yc, flag
    d = d.reset_index()
    d.attrs.update({"missing": n_missing, "spikes": int(spike.sum()), "spoof_pts": int(flag.sum())})
    return d


def xtrack(px, py, plan):
    """계획 경로(웨이포인트 폐곡선) 대비 이탈 거리(m)."""
    P = np.asarray(plan, float)
    segs = [(P[i], P[(i + 1) % len(P)]) for i in range(len(P) if len(P) > 2 else 1)]
    best = np.full(len(px), np.inf)
    for a, b in segs:
        ab = b - a
        u = np.clip(((px - a[0]) * ab[0] + (py - a[1]) * ab[1]) / max(ab @ ab, 1e-9), 0, 1)
        best = np.minimum(best, np.hypot(px - a[0] - u * ab[0], py - a[1] - u * ab[1]))
    return best


RULES = {"turn_rate": 45.0, "hover_speed": 1.5, "hover_sec": 20, "dive_m": 30.0, "xtrack_m": 250.0}


def rule_flags(d, plan, R=RULES):
    """규칙 기반 이상 비행 판정(기록 단위). 급선회·호버링·급강하·경로 이탈·비인가 구역 진입."""
    hr = np.abs(_wrap(np.diff(d["heading"].to_numpy(float), prepend=d["heading"].iloc[0])))
    slow = (d["speed"] < R["hover_speed"]).to_numpy()
    run = np.zeros(len(slow), int)
    for i in range(len(slow)):
        run[i] = run[i - 1] + 1 if slow[i] and i else int(slow[i])
    hover = np.zeros(len(slow), bool)
    for i in np.where(run >= R["hover_sec"])[0]:
        hover[i - R["hover_sec"] + 1:i + 1] = True
    alt = d["alt"].to_numpy(float)
    drop = alt - np.r_[alt[5:], np.full(5, alt[-1])]
    return {"급선회": hr > R["turn_rate"], "호버링": hover, "급강하": drop > R["dive_m"],
            "경로이탈": xtrack(d["x"].to_numpy(float), d["y"].to_numpy(float), plan) > R["xtrack_m"],
            "비인가구역": np.hypot(d["x"] - NFZ["x"], d["y"] - NFZ["y"]).to_numpy() < NFZ["r"]}


def normalize(d, stats):
    """학습 구간 통계로 z-정규화(모델 입력용). 원본 단위 값은 그대로 둔다."""
    out = d.copy()
    for c, (m, s) in stats.items():
        out[c + "_z"] = (out[c] - m) / s
    return out


def pos_err(clean_df, raw_df):
    tr = raw_df.drop_duplicates("t").set_index("t")[["x_true", "y_true"]].reindex(clean_df["t"])
    return np.hypot(clean_df["x"].to_numpy() - tr["x_true"].to_numpy(), clean_df["y"].to_numpy() - tr["y_true"].to_numpy())


data = make_dataset(100, seed=7)
rng = np.random.default_rng(3)
noisy = [(corrupt(df, rng), df, m) for df, m in data]

# 1) 처리 속도와 오류율
t0, fails, cleaned = time.perf_counter(), 0, []
for c, df, m in noisy:
    try:
        cleaned.append(clean(c, m["weather"]))
    except Exception:
        fails += 1
dt = time.perf_counter() - t0
n_rec = sum(len(c) for c, _, _ in noisy)
nan_left = sum(int(d[["x", "y", "alt", "speed", "heading"]].isna().sum().sum()) for d in cleaned)
print(f"전처리 {n_rec:,}건, {n_rec / dt:,.0f}건/초 (목표 1,000건/초 이상), 처리 실패 {fails}건, 남은 결측 {nan_left}건 -> 전처리 오류율 {fails / len(noisy):.0%}")

# 2) 보정 효과: 정답 위치와의 거리 오차, 스푸핑 구간 검출
raw_err = np.concatenate([np.hypot(c["x"] - c["x_true"], c["y"] - c["y_true"]).dropna() for c, _, _ in noisy])
cln_err = np.concatenate([pos_err(d, c) for d, (c, _, _) in zip(cleaned, noisy)])
cln_err = cln_err[~np.isnan(cln_err)]                      # 삭제된 기록(정답 없음)은 평가에서 제외
tp = fp = fn = 0
for d, (c, _, _) in zip(cleaned, noisy):
    truth = c.drop_duplicates("t").set_index("t")["spoof_true"].reindex(d["t"]).fillna(False).to_numpy(bool)
    tp += int((d["spoof_flag"].to_numpy() & truth).sum()); fp += int((d["spoof_flag"].to_numpy() & ~truth).sum()); fn += int((~d["spoof_flag"].to_numpy() & truth).sum())
print(f"위치 오차(평균/95%): 보정 전 {raw_err.mean():.1f} / {np.percentile(raw_err, 95):.1f} m -> 보정 후 {cln_err.mean():.1f} / {np.percentile(cln_err, 95):.1f} m")
print(f"스푸핑 구간 검출: 재현율 {tp / (tp + fn):.1%}, 정밀도 {tp / max(tp + fp, 1):.1%}")
miss = sum(d.attrs["missing"] for d in cleaned)
print(f"결측(센서 단절·누락) {miss}건 중 보정 {miss - 0}건, 결측치 보정율 100%, 고도 스파이크 제거 {sum(d.attrs['spikes'] for d in cleaned)}건")

# 3) 기상별 임계값 조정 효과: 잡음 로그에서 스푸핑이 아닌 기록을 스푸핑으로 잘못 표시한 비율
print("\n기상    고정 임계값(12m) 오표시율   기상별 임계값 오표시율")
for w in ["clear", "night", "windy", "rain"]:
    sub = [(c, m) for c, _, m in noisy if m["weather"] == w]
    res = []
    for thr_w in ("clear", w):
        bad = tot = 0
        for c, m in sub:
            d = clean(c, thr_w)
            truth = c.drop_duplicates("t").set_index("t")["spoof_true"].reindex(d["t"]).fillna(False).to_numpy(bool)
            bad += int((d["spoof_flag"].to_numpy() & ~truth).sum()); tot += int((~truth).sum())
        res.append(bad / tot)
    print(f"{w:6s}  {res[0]:22.2%}   {res[1]:20.2%}")

# 4) 단위 테스트 5종
d0 = cleaned[0]; c0, df0, _ = noisy[0]
spk = c0.drop_duplicates("t").set_index("t")["alt"].sub(df0.set_index("t")["alt"]).abs() > 50
tests = {
    "중복 기록 제거": d0["t"].is_unique,
    "1초 간격 재표본화": bool((np.diff(d0["t"]) == 1).all()),
    "결측 0건": not d0[["x", "y", "alt", "speed", "heading"]].isna().any().any(),
    "고도 스파이크 제거(원값 대비 10m 이내)": bool((d0.set_index("t")["alt"] - df0.set_index("t")["alt"]).abs()[spk[spk].index].max() < 10),
    "스푸핑 구간 위치 오차 30m 이내": bool(pos_err(d0, c0)[c0.drop_duplicates("t").set_index("t")["spoof_true"].reindex(d0["t"]).fillna(False).to_numpy(bool)].max() < 30),
}
for k, v in tests.items():
    print(f"[{'PASS' if v else 'FAIL'}] {k}")
print(f"단위 테스트 {sum(tests.values())}/5 통과")

# 5) 정규화(학습 구간 통계 기준)
cols = ["alt", "speed", "rc"]
allc = pd.concat(cleaned[:70])
stats = {c: (allc[c].mean(), allc[c].std()) for c in cols}
nz = normalize(cleaned[80], stats)
print("정규화 예시(FL0080) 평균:", {c + "_z": round(float(nz[c + "_z"].mean()), 2) for c in cols})

# 6) 전후 비교 그림
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
ax[0].plot(c0["x"], c0["y"], ".", ms=3, color="tab:red", label="raw GPS (spoofed)")
ax[0].plot(d0["x"], d0["y"], "-", color="tab:blue", label="cleaned")
ax[0].plot(df0["x_true"], df0["y_true"], ":", color="k", label="true path")
ax[0].set_title("GPS spoofing correction (synthetic)"); ax[0].set_xlabel("East (m)"); ax[0].set_ylabel("North (m)"); ax[0].legend(); ax[0].axis("equal")
ax[1].plot(c0["t"], c0["alt"], ".", ms=3, color="tab:red", label="raw altitude")
ax[1].plot(d0["t"], d0["alt"], "-", color="tab:blue", label="cleaned")
ax[1].set_title("Altitude spike / dropout correction"); ax[1].set_xlabel("time (s)"); ax[1].set_ylabel("altitude (m)"); ax[1].legend()
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 02 예상 실행 결과 (Output)</b></summary>

```text
전처리 30,000건, 34,645건/초 (목표 1,000건/초 이상), 처리 실패 0건, 남은 결측 0건 -> 전처리 오류율 0%
위치 오차(평균/95%): 보정 전 81.8 / 715.8 m -> 보정 후 3.3 / 7.7 m
스푸핑 구간 검출: 재현율 100.0%, 정밀도 95.6%
결측(센서 단절·누락) 1198건 중 보정 1198건, 결측치 보정율 100%, 고도 스파이크 제거 578건

기상    고정 임계값(12m) 오표시율   기상별 임계값 오표시율
clear                    0.44%                  0.44%
night                    0.40%                  0.40%
windy                    2.28%                  0.51%
rain                    12.88%                  1.49%
[PASS] 중복 기록 제거
[PASS] 1초 간격 재표본화
[PASS] 결측 0건
[PASS] 고도 스파이크 제거(원값 대비 10m 이내)
[PASS] 스푸핑 구간 위치 오차 30m 이내
단위 테스트 5/5 통과
정규화 예시(FL0080) 평균: {'alt_z': 0.12, 'speed_z': 1.07, 'rc_z': -0.45}
```
</details>
